# PART 2. Model Training

- Preparation runs once from notebook 1's saved cleaned tables, without downloads. Only development sites and hours through `FIT_END_LOCAL` are loaded. Validation/test observations stay out of this training sampler; existing evaluation utilities remain separate.
- Each episode rotates the target and chooses multiple other development sites as simultaneous references. All reference slots share physical-variable normalization fitted once on development training data. Metadata unobserved across development sites is omitted. Cached per-site arrays are reused; no reference combinations, histories per task, or manifests are materialized.



- All measured wind is assumed to be at 3 m throughout the study; this constant height is omitted from model predictors.


In [1]:
"""------------------------ IMPORT LIBRARIES AND INITIAL PREP DATA ------------------------"""
# Run ONCE before training: load cleaned outputs and fit development-only normalization
from pathlib import Path
import numpy as np
from wind_pipeline import prepare_training_data, sample_training_episode

DATA_ROOT = Path.cwd() / 'analyses_output_files/wind_pipeline'
WIND_PATH = DATA_ROOT / 'observations/wind.parquet'
ERA5_PATH = DATA_ROOT / 'observations/era5.parquet'
METADATA_PATH = DATA_ROOT / 'metadata/metadata.parquet'  # JSON is also supported
DEVELOPMENT_SITES = ['az06', 'az05', 'az22', 'az26']
VALIDATION_SITES, TEST_SITES = ['az19'], ['az15']
FIT_END_LOCAL = '2015-12-31'
N_REFERENCES = 2
CAMPAIGN_MONTHS = [3]  # optionally [3, 6, 12] calendar months
RANDOM_SEED = 42

prepared_data = prepare_training_data(
    WIND_PATH, ERA5_PATH, METADATA_PATH,
    development_sites=DEVELOPMENT_SITES, validation_sites=VALIDATION_SITES,
    test_sites=TEST_SITES, fit_end_local=FIT_END_LOCAL,
    n_references=N_REFERENCES, campaign_months=CAMPAIGN_MONTHS)
rng = np.random.default_rng(RANDOM_SEED)
FEATURE_NAMES = prepared_data['feature_names']
TARGET_NAMES = prepared_data['target_names']
print('Predictors:', len(FEATURE_NAMES), '| targets:', TARGET_NAMES)
print('Excluded unobserved metadata:', prepared_data['excluded_unobserved_metadata'])


Predictors: 31 | targets: ['target_speed_mps', 'target_direction_sin', 'target_direction_cos']
Excluded unobserved metadata: []


In [2]:
"""------------------------ REPEATED DATA PREPARATION ------------------------"""
# Run at EACH future training step
    # DON'T rerun preparation or reseed the RNG here

episode = sample_training_episode(prepared_data, rng)
for name in ['context_x', 'context_y', 'query_x', 'query_y']:
    print(name, episode[name].shape, '| mask', episode[name + '_mask'].shape)
    assert episode[name].shape == episode[name + '_mask'].shape
    assert np.isfinite(episode[name]).all()
    assert (episode[name][~episode[name + '_mask']] == 0).all()

chosen = [episode['target_site_id'], *episode['reference_site_ids']]
assert len(chosen) == len(set(chosen)) == N_REFERENCES + 1
assert set(chosen).issubset(DEVELOPMENT_SITES)
assert not set(chosen).intersection(VALIDATION_SITES + TEST_SITES)
context_times, query_times = episode['context_times_utc'], episode['query_times_utc']
begin = episode['campaign_start_utc'].tz_localize(None).to_datetime64()
finish = episode['campaign_end_utc'].tz_localize(None).to_datetime64()
cutoff = prepared_data['fit_end_utc'].tz_localize(None).to_datetime64()
first = prepared_data['start_utc'].tz_localize(None).to_datetime64()
assert first <= begin < finish <= cutoff
assert ((context_times > begin) & (context_times <= finish)).all()
assert ((query_times <= begin) | (query_times > finish)).all()
assert (context_times > first).all() and (query_times > first).all()
assert (context_times <= cutoff).all() and (query_times <= cutoff).all()
assert np.intersect1d(context_times, query_times).size == 0
assert np.unique(context_times).size == len(context_times)
assert np.unique(query_times).size == len(query_times)
assert not any(name.startswith(('target_speed', 'target_direction')) for name in FEATURE_NAMES)
print('Target:', chosen[0], '| ordered reference slots:', episode['reference_site_ids'])
print('Campaign UTC boundaries:', episode['campaign_start_utc'], episode['campaign_end_utc'])


context_x (256, 31) | mask (256, 31)
context_y (256, 3) | mask (256, 3)
query_x (256, 31) | mask (256, 31)
query_y (256, 3) | mask (256, 3)
Target: az06 | ordered reference slots: ('az26', 'az22')
Campaign UTC boundaries: 2009-08-28 07:00:00+00:00 2009-11-28 07:00:00+00:00


Each call returns NumPy `context_x (C,F)`, `context_y (C,3)`, `query_x (Q,F)`, `query_y (Q,3)` and matching boolean `_mask` arrays. `C,Q` are up to 256 by default, without replacement or padding. Target order is speed, sin(direction), cos(direction); speed is standardized, direction targets remain unchanged. Missing normalized values are zero with masks; calm speed labels remain valid while undefined directions are masked.

`feature_names` fixes reference-slot order: each reference's measured wind, ERA5, metadata; then target ERA5/metadata and cyclic time once. Measured target wind is used only in `context_y`/`query_y`. IDs, exact UTC boundaries, and sampled UTC `datetime64[ns]` timestamps accompany the arrays.

Campaigns start on randomly chosen eligible **local dates** and end after the selected number of calendar months. Hourly endpoints obey `start_utc < time_utc <= end_utc`. Before subsampling, at least 24 context hours and 50% of campaign hours must have observed target speed, reference speed at every chosen reference, and all target/reference ERA5 components. Outside-campaign query hours meet the same observation requirements. Missing reference direction and metadata retain masks. Thresholds and bounded retries can be configured on `sample_training_episode`; inadequate coverage raises a clear error. This prepares independent query hours, not temporal sliding sequences.
